# PPE zero-shot VLM evaluation — with metrics

This notebook is adapted from your `load_model.ipynb`. It tests **pretrained Ollama vision models** with images, compares their predictions with **human-verified labels**, and calculates **accuracy, precision, recall, F1, false-positive rate, coverage, and processing time**.

**No training or fine-tuning takes place.**

**Important:** The metrics are **image-level**. A violation means *at least one visible worker is missing a helmet or high-visibility vest*. Worker positions and boots are reported, but not scored unless you provide suitable worker-level/boot labels. Models can make mistakes even when they output valid JSON.

### How to use
1. Run **Install** once (if necessary), then **Settings**.
2. Run **Prepare/check verified labels**. If a CSV is missing, a blank review CSV will be created—fill it by checking the images before calculating accuracy.
3. Run **Model helpers**, **Evaluate**, then **Metrics and export**.
4. Change `MODEL_CHOICE` and rerun **Evaluate** + **Metrics and export** to compare models.

To reuse the **PPE_Balanced_VLM_Evaluation** package from earlier, point `IMAGE_DIR` at its `test/images` folder and `LABELS_CSV` at its *verified, exported* `ground_truth_review.csv`.

## 1. Install libraries (first time only)

In [1]:
%pip install -U ollama pandas scikit-learn matplotlib

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
    --------------------------------------- 0.3/10.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.9 MB 909.9 kB/s eta 0:00:12
   --- ------------------------------------ 1.0/10.9 MB 1.6 MB/s eta 0:00:07
   ----- ---------------------------------- 1.6/10.9 MB 1.8 MB/s eta 0:00:06
   ------- -------------------------------- 2.1/10.9 MB 2.1 MB/s eta 0:00:05
   ---------- ----------------------------- 2.9/10.9 MB 2.4 MB/s eta 0:00:04
   ------------ --------------------------- 3.4/10.9 MB 2.4 MB/s eta 0:00:04
   -------------- ------------------------- 3.9/10.9 MB 2.6 MB/s eta 0:00:03
   ---------------- ----------------------- 4.5/10.9 MB 2.4 MB/s eta 0:00:03
   ------------------ --------------------- 5.0/10.9 MB 2.4 MB/s eta 0:00:03
   -------------------- ------------------- 5.5/10.9 MB 2.6 MB/s eta 0:00:03
   -------


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Settings

Change the paths to match your computer. `RUN_LIMIT = 5` tests five **verified** images first; set to `0` to process all verified images. Choose `MODE = "cloud"` only if the model is actually offered by your remote Ollama provider. Models not listed there cannot be invoked remotely simply by changing the model name.

In [3]:
import sys
print(sys.executable)

c:\python315\python.exe


In [9]:
%pip show pandas

Name: pandas
Version: 3.1.0
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License-Expression: BSD-3-Clause
Location: C:\Users\GYS\AppData\Roaming\Python\Python315\site-packages
Requires: numpy, python-dateutil, tzdata
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path
from getpass import getpass
from ollama import Client
import json, re, time
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display

# Model options: use the exact names shown by `ollama list`, or the cloud catalog.
MODEL_CHOICE = "openbmb/minicpm-v4.7:1b"
MODE = "local"                 # "local" or "cloud"
RUN_LIMIT = 5                  # 0 = evaluate ALL verified images
REQUEST_TIMEOUT = 300         # seconds per request
SHOW_IMAGES = False           # True = show thumbnails during evaluation

IMAGE_DIR = Path(r"C:\Users\GYS\Downloads\test model\model testing images")
LABELS_CSV = IMAGE_DIR / "ground_truth_review.csv"

# To use the earlier balanced evaluation package instead, uncomment and edit:
# DATASET_DIR = Path(r"C:\Users\GYS\Downloads\PPE_Balanced_VLM_Evaluation")
# IMAGE_DIR = DATASET_DIR / "test" / "images"
# LABELS_CSV = DATASET_DIR / "ground_truth_review.csv"

RESULTS_DIR = Path.cwd() / "ppe_model_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if MODE == "cloud":
    key = getpass("Ollama Cloud API key: ")
    client = Client(host="https://ollama.com", headers={"Authorization": f"Bearer {key}"}, timeout=REQUEST_TIMEOUT)
else:
    client = Client(host="http://localhost:11434", timeout=REQUEST_TIMEOUT)

print(f"Model: {MODEL_CHOICE} | Mode: {MODE}")
print(f"Images: {IMAGE_DIR}")
print(f"Labels: {LABELS_CSV}")

ImportError: DLL load failed while importing missing: An Application Control policy has blocked this file.

## 3. Prepare / check verified ground truth

You must visually check the images against the dataset owner's annotations. Use `yes` when **at least one worker visibly lacks** the item, `no` when **all assessable workers wear it** and there are no unassessable workers, and `unclear` when the evidence is uncertain. Set `review_status` to `verified` only after checking. Never mark images compliant solely because no missing-PPE annotation exists.

If a blank CSV is created for your own image folder, edit it in Excel and save **as CSV**, retaining the column names. The notebook supports the earlier balanced package's `verified_helmet` / `verified_vest` fields and the starter kit's `verified_label` (overall-only).

In [ ]:
from IPython.display import Image as NotebookImage

if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Image directory not found: {IMAGE_DIR}. Edit IMAGE_DIR in Settings.")

all_images = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
if not all_images:
    raise ValueError(f"No JPG/PNG images found in: {IMAGE_DIR}")

if not LABELS_CSV.exists():
    # Creates a review sheet, NOT invented ground truth.
    pd.DataFrame({
        'image': [p.name for p in all_images],
        'verified_helmet': [''] * len(all_images),
        'verified_vest': [''] * len(all_images),
        'review_status': ['needs_review'] * len(all_images),
        'notes': [''] * len(all_images)
    }).to_csv(LABELS_CSV, index=False, encoding='utf-8-sig')
    print(f"Created blank review CSV: {LABELS_CSV}")
    print("Open it in Excel, verify the photos, then mark verified labels before evaluating.")

labels = pd.read_csv(LABELS_CSV, dtype=str, keep_default_na=False, encoding='utf-8-sig')
if 'image' not in labels.columns or 'review_status' not in labels.columns:
    raise ValueError("Ground-truth CSV needs 'image' and 'review_status' columns.")

for field in ['verified_helmet', 'verified_vest', 'verified_label']:
    if field in labels:
        labels[field] = labels[field].astype(str).str.strip().str.lower()
labels['review_status'] = labels['review_status'].str.strip().str.lower()

if {'verified_helmet', 'verified_vest'} <= set(labels.columns):
    data_type = 'per_item'
    selected_labels = labels[(labels.review_status == 'verified') &
        labels.verified_helmet.isin(['yes','no','unclear']) &
        labels.verified_vest.isin(['yes','no','unclear'])].copy()
elif 'verified_label' in labels.columns:
    data_type = 'overall_only'
    selected_labels = labels[(labels.review_status == 'verified') &
        labels.verified_label.isin(['violation','compliant','unclear'])].copy()
else:
    raise ValueError("Need verified_helmet + verified_vest, or verified_label, in the ground-truth CSV.")

selected_labels = selected_labels[selected_labels.image.apply(lambda n: (IMAGE_DIR / n).is_file())]
print(f"Found {len(all_images)} image files; {len(labels)} CSV entries; {len(selected_labels)} verified and available.")
print("Label format:", data_type)
if len(selected_labels) == 0:
    print("NO VERIFIED IMAGES YET: inspect photos and complete the CSV before running evaluation.")
else:
    display(selected_labels.head(6))

## 4. Prompt and parsing helpers

For fair comparisons, use exactly the same image and prompt for every model. The prompt requests JSON so Python can measure *actual predictions* rather than guess from free-form text. The script considers an invalid response an `error`, not a compliant image.

In [ ]:
PROMPT = """Assess this construction-site image for PPE compliance.
Inspect every visible worker for a safety helmet and a high-visibility vest.
Also describe visible safety boots, but do NOT infer certified safety footwear from appearance alone.

For helmet/vest classifications: yes = at least one worker is clearly missing it;
no = all visible workers can be assessed and none are missing it;
unclear = at least one worker cannot be assessed, and no definite absence is visible.
Never guess when the relevant body part is obstructed.

Return ONLY valid JSON, with exactly these fields and permitted values:
{
  "worker_count": 2,
  "workers": [
    {"position": "left", "helmet": "worn", "vest": "missing", "boots": "unclear"}
  ],
  "helmet_violation": "no",
  "vest_violation": "yes",
  "boots_violation": "unclear"
}
For every violation field, choose only one of: yes, no, unclear. The JSON above is an example of formatting, NOT the expected answer.
"""

def overall_from_items(helmet, vest):
    """Image-level decision from helmet/vest only (boots not in ground truth)."""
    if 'yes' in (helmet, vest):
        return 'violation'
    if helmet == 'no' and vest == 'no':
        return 'compliant'
    return 'unclear'

def parse_vlm_response(raw):
    """Parse valid JSON; explicitly report noncompliant/unparseable responses."""
    raw = str(raw or '').strip()
    raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw, flags=re.I)
    try:
        obj = json.loads(raw)
    except (ValueError, TypeError):
        left, right = raw.find('{'), raw.rfind('}')
        if left < 0 or right <= left:
            return None
        try:
            obj = json.loads(raw[left:right + 1])
        except (ValueError, TypeError):
            return None
    if not isinstance(obj, dict):
        return None
    for field in ('helmet_violation', 'vest_violation'):
        v = obj.get(field)
        if not isinstance(v, str) or v.strip().lower() not in ('yes', 'no', 'unclear'):
            return None
        obj[field] = v.strip().lower()
    boots = str(obj.get('boots_violation', 'unclear')).strip().lower()
    obj['boots_violation'] = boots if boots in ('yes','no','unclear') else 'unclear'
    return obj

def truth_for_row(row):
    if data_type == 'per_item':
        h, v = row['verified_helmet'], row['verified_vest']
        return h, v, overall_from_items(h, v)
    overall = row['verified_label']
    return '', '', overall

# Small offline sanity checks: no Ollama call or model download.
assert overall_from_items('no','no') == 'compliant'
assert overall_from_items('yes','unclear') == 'violation'
assert overall_from_items('no','unclear') == 'unclear'
assert parse_vlm_response('{"helmet_violation":"yes","vest_violation":"no"}') is not None
print('Parsing and ground-truth helpers are ready.')

## 5. Analyse the verified images

You can reuse this section for local **or remote** models by changing Settings. Each model is assessed **one image at a time**, and its raw response and error status are saved. A model that refuses, fails, or returns invalid JSON is recorded, not silently marked compliant.

In [ ]:
if selected_labels.empty:
    raise ValueError("No verified labels available. Complete ground_truth_review.csv, rerun section 3, then try again.")

run_rows = selected_labels.head(RUN_LIMIT) if RUN_LIMIT > 0 else selected_labels
predictions = []
run_started = time.perf_counter()

for i, (_, row) in enumerate(run_rows.iterrows(), start=1):
    path = IMAGE_DIR / row['image']
    truth_h, truth_v, truth_overall = truth_for_row(row)
    print(f"[{i}/{len(run_rows)}] {path.name}")
    if SHOW_IMAGES:
        display(NotebookImage(filename=str(path), width=320))

    started = time.perf_counter()
    raw, parsed, error = '', None, ''
    try:
        response = client.chat(
            model=MODEL_CHOICE,
            messages=[{'role': 'user', 'content': PROMPT, 'images': [path.read_bytes()]}],
            options={'temperature': 0}
        )
        raw = response.message.content or ''
        parsed = parse_vlm_response(raw)
        if parsed is None:
            error = 'Unparseable JSON or invalid helmet/vest values'
    except Exception as exc:
        error = f'{type(exc).__name__}: {exc}'

    seconds = time.perf_counter() - started
    predicted_h = parsed['helmet_violation'] if parsed else 'error'
    predicted_v = parsed['vest_violation'] if parsed else 'error'
    predicted_o = overall_from_items(predicted_h, predicted_v) if parsed else 'error'
    predictions.append({
        'image': path.name,
        'model': MODEL_CHOICE,
        'ground_truth': truth_overall,
        'predicted': predicted_o,
        'truth_helmet': truth_h,
        'pred_helmet': predicted_h,
        'truth_vest': truth_v,
        'pred_vest': predicted_v,
        'pred_boots': parsed['boots_violation'] if parsed else 'error',
        'pred_worker_count': parsed.get('worker_count', '') if parsed else '',
        'seconds': round(seconds, 3),
        'error': error,
        'raw_response': raw,
    })
    print(f"  Ground truth: {truth_overall} | Prediction: {predicted_o} | {seconds:.2f}s")
    if error:
        print(f"  ERROR: {error}")

total_seconds = time.perf_counter() - run_started
result_df = pd.DataFrame(predictions)
print(f"\nAll images processed! {len(result_df)} images | {total_seconds:.2f}s total wall time")
display(result_df[['image','ground_truth','predicted','seconds','error']].head(10))

## 6. Compute evaluation metrics and confusion matrix

**Important interpretation:** Standard accuracy, precision, recall, and F1 below are calculated only on **decisive** (`compliant`/`violation`) predictions with known binary ground truth. We ALSO show **coverage**, **strict accuracy**, and **actionable recall**, so a model cannot look artificially good merely by answering `unclear` or producing errors.

- **Coverage:** fraction of verified binary images receiving decisive predictions.
- **Strict accuracy:** fraction of all verified binary images correctly predicted, counting `unclear` and errors as incorrect.
- **Actionable recall:** percentage of actual violations that triggered a `violation` prediction; `unclear` and errors do not count as detections.
- `No data` means no denominator exists (for example, recall with zero actual violations).

In [ ]:
def metric_report(data, truth_col, prediction_col, positive='violation', negative='compliant'):
    """Binary metrics + coverage and abstention-aware measures."""
    labeled = data[data[truth_col].isin([positive,negative])].copy()
    decided = labeled[labeled[prediction_col].isin([positive,negative])]

    tp = int(((decided[truth_col] == positive) & (decided[prediction_col] == positive)).sum())
    tn = int(((decided[truth_col] == negative) & (decided[prediction_col] == negative)).sum())
    fp = int(((decided[truth_col] == negative) & (decided[prediction_col] == positive)).sum())
    fn = int(((decided[truth_col] == positive) & (decided[prediction_col] == negative)).sum())

    safe_div = lambda n, d: (n / d if d else None)
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * tp, 2 * tp + fp + fn)
    total_positive = int((labeled[truth_col] == positive).sum())

    return {
        'ground_truth_binary': len(labeled),
        'scored_decisive': len(decided),
        'unclear_or_error_on_binary': len(labeled) - len(decided),
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'accuracy': safe_div(tp + tn, len(decided)),
        'precision': precision, 'recall': recall, 'f1': f1,
        'false_positive_rate': safe_div(fp, fp + tn),
        'coverage': safe_div(len(decided), len(labeled)),
        'strict_accuracy': safe_div(tp + tn, len(labeled)),
        'actionable_recall': safe_div(tp, total_positive),
    }

# Unit check using synthetic outcomes (not actual PPE performance)
_test = pd.DataFrame({'gold':['violation','compliant','violation','compliant'],
                      'pred':['violation','violation','unclear','compliant']})
_check = metric_report(_test,'gold','pred')
assert (_check['tp'],_check['tn'],_check['fp'],_check['fn']) == (1,1,1,0)
assert abs(_check['strict_accuracy'] - 0.5) < 1e-9
assert abs(_check['actionable_recall'] - 0.5) < 1e-9

reports = {'overall': metric_report(result_df, 'ground_truth','predicted')}
if data_type == 'per_item':
    reports['helmet'] = metric_report(result_df,'truth_helmet','pred_helmet','yes','no')
    reports['vest'] = metric_report(result_df,'truth_vest','pred_vest','yes','no')

report_df = pd.DataFrame.from_dict(reports, orient='index')
print(f'Model: {MODEL_CHOICE} | Images analysed: {len(result_df)}')
print(f'Average seconds/image: {result_df.seconds.mean():.2f} | Total model-request seconds: {result_df.seconds.sum():.2f}')
display(report_df.round(3))

# Confusion matrix excludes uncertain/errors; coverage and strict accuracy quantify exclusions.
scorable = result_df[result_df.ground_truth.isin(['compliant','violation']) &
                     result_df.predicted.isin(['compliant','violation'])]
if not scorable.empty:
    labels_order = ['compliant','violation']
    cm = confusion_matrix(scorable.ground_truth, scorable.predicted, labels=labels_order)
    fig, ax = plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay(cm, display_labels=labels_order).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{MODEL_CHOICE} — decisive image-level predictions')
    plt.tight_layout()
    plt.show()
else:
    print('No decisive, binary-labelled images: confusion matrix unavailable.')

print('NOTE: "No data" (NaN) is expected if a class/denominator is absent in this sample.')

## 7. Export results and update model comparison

Exports a detailed CSV with predictions, errors and raw responses, a metrics CSV, and an ongoing comparison CSV. Rerunning the same model replaces its comparison row. Files are saved in `ppe_model_results` beside your notebook (working directory).

In [ ]:
safe_name = re.sub(r'[^a-zA-Z0-9_-]+', '_', MODEL_CHOICE)
results_path = RESULTS_DIR / f'predictions_{safe_name}.csv'
metrics_path = RESULTS_DIR / f'metrics_{safe_name}.csv'
comparison_path = RESULTS_DIR / 'model_comparison.csv'

result_df.to_csv(results_path, index=False, encoding='utf-8-sig')
report_df.to_csv(metrics_path, encoding='utf-8-sig', index_label='scope')

new_rows = []
for scope, metrics in reports.items():
    new_rows.append({
        'model': MODEL_CHOICE,
        'scope': scope,
        'images_analyzed': len(result_df),
        'avg_seconds': round(result_df.seconds.mean(),3),
        'total_request_seconds': round(result_df.seconds.sum(),3),
        **metrics,
    })
comparison = pd.DataFrame(new_rows)
if comparison_path.exists():
    existing = pd.read_csv(comparison_path)
    existing = existing[~((existing['model'] == MODEL_CHOICE) & (existing['scope'].isin(comparison.scope)))]
    comparison = pd.concat([existing, comparison], ignore_index=True)
comparison.to_csv(comparison_path, index=False, encoding='utf-8-sig')

print('Saved:', results_path)
print('Saved:', metrics_path)
print('Updated:', comparison_path)
display(comparison[['model','scope','accuracy','precision','recall','f1','coverage','strict_accuracy','actionable_recall','avg_seconds']].round(3))

### Reading your results

- If your five images contain **only violations**, you cannot meaningfully estimate the false-positive rate; include compliant images.
- If there are **no verified labels**, the notebook will not invent accuracy.
- Results on a small sample are preliminary; for your report, verify a larger balanced set and reuse the same test images across all models.
- A higher model score with very low **coverage** may be misleading; inspect the failures and unclear predictions.
- Boots remain descriptive only. To calculate accuracy for safety boots, you'll need separately verified boot labels.
- For a safety application, manager review is still necessary before treating an AI detection as a confirmed incident.